# 04 – Train U-Net: Lane Segmentation

**CSE445 – Road Damage Detection & Lane Segmentation**

This notebook:
1. Loads train / val split manifests from `data/lane/splits/`
2. Builds `SegmentationDataset` + `DataLoader`
3. Instantiates U-Net + Adam + `BCEDiceLoss`
4. Trains with early stopping + LR scheduling
5. Plots learning curves

**Prerequisites**: Run `02_EDA_lane.ipynb` first (generates split CSVs).

In [ ]:
import sys, os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/Road_Damage_Project'
    os.environ['RUN_ENV'] = 'colab'
except ImportError:
    REPO = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
    os.environ['RUN_ENV'] = 'local'

if REPO not in sys.path:
    sys.path.insert(0, REPO)
print('Repo root:', REPO)

In [ ]:
import torch
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

import config
from src.shared.dataset    import SegmentationDataset
from src.shared.transforms import get_train_transforms, get_val_transforms
from src.shared.unet       import UNet
from src.shared.losses     import BCEDiceLoss
from src.shared.trainer    import Trainer

print('CUDA available:', torch.cuda.is_available())

## 1. Dataset & DataLoaders

In [ ]:
cfg = config.LANE_UNET
img_size = (config.IMG_HEIGHT, config.IMG_WIDTH)

train_ds = SegmentationDataset(
    config.LANE_SPLIT_DIR / 'train.csv',
    transform=get_train_transforms(img_size)
)
val_ds = SegmentationDataset(
    config.LANE_SPLIT_DIR / 'val.csv',
    transform=get_val_transforms(img_size)
)

train_loader = DataLoader(train_ds, batch_size=cfg['batch_size'], shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=cfg['batch_size'], shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds)} samples  |  Val: {len(val_ds)} samples')

## 2. Model, Loss, Trainer

In [ ]:
model   = UNet(cfg['in_channels'], cfg['out_channels'], cfg['base_features'])
loss_fn = BCEDiceLoss(cfg['bce_weight'], cfg['dice_weight'])
trainer = Trainer(model, loss_fn, cfg, train_loader, val_loader)

print(f'U-Net parameters : {model.count_parameters():,}')
print(f'Experiment dir   : {trainer.exp_dir}')

## 3. Train

In [ ]:
history = trainer.fit(n_epochs=None)

## 4. Learning Curves

In [ ]:
df = pd.read_csv(trainer.log_csv_path)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('U-Net Lane Segmentation – Learning Curves', fontsize=13, fontweight='bold')

axes[0].plot(df['epoch'], df['train_loss'], label='Train', color='#1565C0', lw=2)
axes[0].plot(df['epoch'], df['val_loss'],   label='Val',   color='#E53935', lw=2, ls='--')
axes[0].set_title('BCEDice Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(df['epoch'], df['train_iou'], label='Train', color='#1565C0', lw=2)
axes[1].plot(df['epoch'], df['val_iou'],   label='Val',   color='#E53935', lw=2, ls='--')
axes[1].set_title('IoU'); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(df['epoch'], df['train_dice'], label='Train', color='#1565C0', lw=2)
axes[2].plot(df['epoch'], df['val_dice'],   label='Val',   color='#E53935', lw=2, ls='--')
axes[2].set_title('Dice Score'); axes[2].legend(); axes[2].grid(alpha=0.3)

for ax in axes: ax.set_xlabel('Epoch')
plt.tight_layout()
plt.savefig(trainer.exp_dir / 'curves.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Hyperparameter Notes

To experiment with different settings, edit `config.LANE_UNET` in `config.py`:

| Parameter | Default | Notes |
|---|---|---|
| `lr` | 1e-4 | Try 5e-4 for faster initial learning |
| `base_features` | 32 | Try 64 for more capacity |
| `batch_size` | 8 | Increase to 16 if GPU has >8 GB VRAM |
| `bce_weight` | 0.5 | Try 0.3/0.7 (dice-heavy) for imbalanced data |
| `run_name` | lane_unet_run1 | Change to lane_unet_run2 for new experiment |

Each run saves its own config snapshot and checkpoint in `experiments/`.

**Next**: `05_evaluate_crack.ipynb`